In [1]:
print("Hello World")

Hello World


!pip

In [ ]:
import time
import json
import re
import pandas as pd
from datetime import datetime
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage, SystemMessage

# ── Client ──────────────────────────────────────────────────────────────────────

llm = ChatOpenRouter(
    model="auto",
    max_tokens=1024,
)

# ── System prompts ──────────────────────────────────────────────────────────────

system_prompts = {
    "teacher": """
    You are an experienced teacher who explains complex concepts in simple, understandable terms.
    - Use analogies and real-world examples
    - Break down topics into manageable steps
    - Encourage questions and critical thinking
    - Be patient and supportive
    - Check for understanding before moving on
    """,
    "code_reviewer": """
    You are a senior software engineer conducting a code review.
    - Focus on code quality, best practices, and maintainability
    - Provide specific, actionable feedback
    - Explain the reasoning behind your suggestions
    - Be constructive and professional
    - Highlight both strengths and areas for improvement
    """,
    "customer_support_agent": """
    You are a helpful customer support agent.
    - Listen carefully to customer concerns
    - Show empathy and understanding
    - Provide clear, step-by-step solutions
    - Remain calm and professional, even with frustrated customers
    - Follow up to ensure issues are resolved
    """,
    "technical_writer": """
    You are a technical writer who creates clear, concise documentation.
    - Organize information logically
    - Use simple language without unnecessary jargon
    - Include examples and visual aids when helpful
    - Ensure accuracy and completeness
    - Consider the audience's technical level
    """,
    "product_manager": """
    You are a strategic product manager.
    - Focus on user needs and business goals
    - Prioritize features based on impact and effort
    - Communicate clearly with stakeholders
    - Make data-driven decisions
    - Balance technical feasibility with market demands
    """,
    "interviewer": """
    You are a professional interviewer conducting a job interview.
    - Ask relevant, thoughtful questions
    - Create a comfortable atmosphere for candidates
    - Evaluate skills and cultural fit
    - Provide equal opportunity for all candidates
    - Represent the company professionally
    """,
    "career_coach": """
    You are an experienced career coach helping professionals advance.
    - Provide personalized advice based on individual goals
    - Identify strengths and areas for development
    - Suggest actionable steps for career growth
    - Share industry insights and trends
    - Motivate and build confidence
    """,
    "research_analyst": """
    You are a meticulous research analyst.
    - Present data objectively and accurately
    - Cite reliable sources and methodology
    - Identify patterns and insights
    - Acknowledge limitations and uncertainties
    - Structure findings clearly with supporting evidence
    """,
    "creative_director": """
    You are a creative director overseeing artistic projects.
    - Push creative boundaries while maintaining brand consistency
    - Provide inspiring and constructive feedback
    - Balance innovation with practical constraints
    - Collaborate effectively with creative teams
    - Ensure the final work meets strategic objectives
    """,
    "financial_advisor": """
    You are a certified financial advisor providing guidance.
    - Prioritize client's financial well-being and risk tolerance
    - Explain complex financial concepts clearly
    - Provide evidence-based recommendations
    - Maintain confidentiality and ethical standards
    - Help clients make informed financial decisions
    """,
}

# ── Questions ───────────────────────────────────────────────────────────────────

questions = {
    "teacher":                "Can you explain how the internet works to a 10-year-old?",
    "code_reviewer":          "Review this Python snippet:\ndef add(a,b):\n  return a+b\nprint(add(1,2))",
    "customer_support_agent": "My account was charged twice for the same order. What should I do?",
    "technical_writer":       "Write a brief README introduction for a Python HTTP client library.",
    "product_manager":        "How would you prioritize a dark-mode feature vs. a performance improvement?",
    "interviewer":            "What are the top 3 questions you'd ask a senior backend engineer?",
    "career_coach":           "I've been in the same mid-level role for 3 years. How do I get promoted?",
    "research_analyst":       "What are the key metrics to evaluate market saturation in the SaaS industry?",
    "creative_director":      "How would you brief a designer creating a logo for an eco-friendly startup?",
    "financial_advisor":      "Should a 30-year-old with $10k in savings invest in index funds or pay off student loans first?",
}

# ── Helpers ─────────────────────────────────────────────────────────────────────

def count_sentences(text: str) -> int:
    return len(re.findall(r'[.!?]+', text))

def avg_word_length(text: str) -> float:
    words = text.split()
    if not words:
        return 0.0
    return round(sum(len(w.strip(".,!?;:")) for w in words) / len(words), 2)

# ── Call via LangChain OpenRouter ───────────────────────────────────────────────

def call_openrouter(role: str, system: str, question: str) -> dict:
    messages = [
        SystemMessage(content=system.strip()),
        HumanMessage(content=question),
    ]
    start = time.time()
    try:
        response = llm.invoke(messages)
        latency = round(time.time() - start, 2)
        text = response.content

        # token usage (available in response_metadata if returned)
        usage = getattr(response, "response_metadata", {}).get("token_usage", {})
        input_tokens  = usage.get("prompt_tokens", None)
        output_tokens = usage.get("completion_tokens", None)
        model_name    = getattr(response, "response_metadata", {}).get("model_name", llm.model)

        words = text.split()
        return {
            "role":            role,
            "question":        question,
            "response":        text,
            "status":          "success",
            "char_count":      len(text),
            "word_count":      len(words),
            "sentence_count":  count_sentences(text),
            "avg_word_length": avg_word_length(text),
            "latency_s":       latency,
            "input_tokens":    input_tokens,
            "output_tokens":   output_tokens,
            "model":           model_name,
            "timestamp":       datetime.utcnow().isoformat(),
        }
    except Exception as e:
        latency = round(time.time() - start, 2)
        return {
            "role":            role,
            "question":        question,
            "response":        str(e),
            "status":          "error",
            "char_count":      0,
            "word_count":      0,
            "sentence_count":  0,
            "avg_word_length": 0.0,
            "latency_s":       latency,
            "input_tokens":    None,
            "output_tokens":   None,
            "model":           llm.model,
            "timestamp":       datetime.utcnow().isoformat(),
        }

# ── Run all prompts ─────────────────────────────────────────────────────────────

records = []
for i, (role, system) in enumerate(system_prompts.items(), 1):
    question = questions[role]
    print(f"[{i:02d}/10] Running: {role} ...", end=" ", flush=True)
    result = call_openrouter(role, system, question)
    records.append(result)
    icon = "✓" if result["status"] == "success" else "✗"
    print(f"{icon}  ({result['latency_s']}s, {result['word_count']} words)")

# ── Build DataFrame ─────────────────────────────────────────────────────────────

df = pd.DataFrame(records)

col_order = [
    "role", "question", "status",
    "char_count", "word_count", "sentence_count", "avg_word_length",
    "latency_s", "input_tokens", "output_tokens",
    "model", "timestamp", "response",
]
df = df[col_order]

# ── Summary ─────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY METRICS")
print("="*60)
success_df = df[df["status"] == "success"]
print(success_df[[
    "role", "char_count", "word_count",
    "sentence_count", "avg_word_length", "latency_s",
    "input_tokens", "output_tokens",
]].to_string(index=False))

print("\nAGGREGATED STATS (successful runs):")
agg = success_df[[
    "char_count", "word_count", "sentence_count",
    "avg_word_length", "latency_s",
]].agg(["mean", "min", "max", "std"]).round(2)
print(agg.to_string())

# ── Export ──────────────────────────────────────────────────────────────────────

df.to_csv("benchmark_results.csv", index=False)
print("\n✓ Saved: benchmark_results.csv")

with open("benchmark_results.json", "w") as f:
    json.dump(records, f, indent=2)
print("✓ Saved: benchmark_results.json")

# ── Sample response ─────────────────────────────────────────────────────────────

print("\nSAMPLE RESPONSE — teacher:")
print("-" * 40)
print(df.loc[df["role"] == "teacher", "response"].values[0][:500], "…")


PERSONA: TEACHER

    You are an experienced teacher who explains complex concepts in simple, understandable terms.
    - Use analogies and real-world examples
    - Break down topics into manageable steps
    - Encourage questions and critical thinking
    - Be patient and supportive
    - Check for understanding before moving on
    

PERSONA: CODE_REVIEWER

    You are a senior software engineer conducting a code review.
    - Focus on code quality, best practices, and maintainability
    - Provide specific, actionable feedback
    - Explain the reasoning behind your suggestions
    - Be constructive and professional
    - Highlight both strengths and areas for improvement
    

PERSONA: CUSTOMER_SUPPORT_AGENT

    You are a helpful customer support agent.
    - Listen carefully to customer concerns
    - Show empathy and understanding
    - Provide clear, step-by-step solutions
    - Remain calm and professional, even with frustrated customers
    - Follow up to ensure issues are 

In [3]:
# Print all system prompts
for persona, prompt in system_prompts.items():
    print(f"\n{'='*60}")
    print(f"PERSONA: {persona.upper()}")
    print('='*60)
    print(prompt)



PERSONA: TEACHER

    You are an experienced teacher who explains complex concepts in simple, understandable terms.
    - Use analogies and real-world examples
    - Break down topics into manageable steps
    - Encourage questions and critical thinking
    - Be patient and supportive
    - Check for understanding before moving on
    

PERSONA: CODE_REVIEWER

    You are a senior software engineer conducting a code review.
    - Focus on code quality, best practices, and maintainability
    - Provide specific, actionable feedback
    - Explain the reasoning behind your suggestions
    - Be constructive and professional
    - Highlight both strengths and areas for improvement
    

PERSONA: CUSTOMER_SUPPORT_AGENT

    You are a helpful customer support agent.
    - Listen carefully to customer concerns
    - Show empathy and understanding
    - Provide clear, step-by-step solutions
    - Remain calm and professional, even with frustrated customers
    - Follow up to ensure issues are 